# 02 - nn.Module, Loss & Optimizer: the real training loop
Goal: build a model class, understand what CrossEntropyLoss and the optimizer actually do,
and assemble the canonical PyTorch training loop.

## nn.Module - packaging layers
A model is a class inheriting from `nn.Module`:
- `__init__` declares the layers → their W and b auto-register as parameters (`requires_grad=True` already set)
- `forward()` defines how data flows through the layers
We call `model(x)`, never `forward()` directly.

In [2]:
import torch
import torch.nn as nn

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)   # layer 1: 784 pixels in → 128 features out
        self.fc2 = nn.Linear(128, 7)     # layer 2: 128 features in → 7 class scores out

    def forward(self, x):
        x = x.reshape(x.size(0), -1)     # flatten each image to a 784 vector
        x = torch.relu(self.fc1(x))      # layer 1, then ReLU
        return self.fc2(x)               # layer 2 → raw scores (logits)

model = TinyNet()
print(model)

TinyNet(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)


## The model
`TinyNet`: two fully-connected layers, 784 → 128 → 7.
- `__init__` declares the layers; `super().__init__()` must come first (enables parameter tracking).
- `forward` = flatten → fc1 → ReLU → fc2 → 7 raw scores (logits).

## Why ReLU
`relu(v) = max(0, v)` - keeps positives, zeros negatives.
Without a nonlinearity between layers, two Linear layers collapse into one (matrix × matrix = matrix),
so depth adds nothing. ReLU breaks the linearity → lets the network learn complex patterns.
*Interview classic: "why do we need activation functions?"*

In [3]:
v = torch.tensor([-2.0, -0.5, 0.0, 1.5, 3.0])
print(torch.relu(v))    # negatives → 0, positives unchanged

tensor([0.0000, 0.0000, 0.0000, 1.5000, 3.0000])


## CrossEntropyLoss - turning wrongness into a number
Takes raw logits (batch, 7) + true labels (batch,), and internally:
1. softmax → probabilities that sum to 1
2. reads the probability given to the TRUE class
3. loss = −log(that prob) → right & confident ≈ 0, wrong & confident → large
Model has NO softmax because the loss does it (avoids double-softmax, more numerically stable).
Dtypes: logits `float32` (batch, classes); labels `int64` (batch,).

In [4]:
criterion = nn.CrossEntropyLoss()

logits = torch.tensor([[2.0, 0.5, -1.0]])   # model strongly favours class 0
label_right = torch.tensor([0])             # true class IS 0
label_wrong = torch.tensor([2])             # pretend true class is 2

print(criterion(logits, label_right))   # should be small
print(criterion(logits, label_wrong))   # should be large

tensor(0.2413)
tensor(3.2413)


## Loss reacts to correctness
Same logits (model favours class 0): loss is small when the true label is 0, large when it's 2.

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [6]:
# fake data for now: 256 random "images" + random labels 0–6
X = torch.randn(256, 1, 28, 28)
y = torch.randint(0, 7, (256,))          # int64 labels - exactly what the loss wants

for epoch in range(30):
    logits = model(X)                    # 1. forward - run data through the desk
    loss = criterion(logits, y)          # 2. how wrong?
    optimizer.zero_grad()                # 3. wipe old gradients (accumulation trap)
    loss.backward()                      # 4. compute gradients for ALL dials, one call
    optimizer.step()                     # 5. turn every dial downhill
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  loss {loss.item():.4f}")

epoch  0  loss 1.9791
epoch  5  loss 1.2894
epoch 10  loss 0.7890
epoch 15  loss 0.4287
epoch 20  loss 0.2082
epoch 25  loss 0.0974


## What just happened - overfitting
Labels were RANDOM, yet training loss fell to ~0.1. The model didn't learn a pattern - it MEMORISED
256 examples (it has enough capacity to act as a lookup table).
**Lesson: low training loss ≠ good model.** This is why we need a held-out validation set (Phase 1) -
train loss is the model grading its own homework; validation is the real exam.

### Ex 1 - how many parameters?
By hand: fc1 = 784×128 + 128 (weights + biases); fc2 = 128×7 + 7. Predict the total, then check.

In [7]:
total = 0
for name, p in model.named_parameters():
    print(name, tuple(p.shape), p.numel())
    total += p.numel()
print("total:", total)

fc1.weight (128, 784) 100352
fc1.bias (128,) 128
fc2.weight (7, 128) 896
fc2.bias (7,) 7
total: 101383


### Ex 2 - remove zero_grad
Loss curve looked ~the same, even slightly lower - but that's misleading:
- The bug is real (gradients accumulate), but **Adam normalises gradient magnitude, masking it**.
- With plain SGD the loss would visibly diverge/spike.
- Also: on memorised random noise, "lower training loss" means nothing - only validation (Phase 1) gives a real verdict.

In [8]:
model = TinyNet()                                    # fresh desk
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

X = torch.randn(256, 1, 28, 28)
y = torch.randint(0, 7, (256,))

for epoch in range(30):
    logits = model(X)
    loss = criterion(logits, y)
    # optimizer.zero_grad()          # ← DELIBERATELY DISABLED
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  loss {loss.item():.4f}")

epoch  0  loss 1.9879
epoch  5  loss 1.2849
epoch 10  loss 0.7787
epoch 15  loss 0.4082
epoch 20  loss 0.1753
epoch 25  loss 0.0603


### Ex 3 - wrong label dtype
Labels must be int64. Feed float labels instead and read the error.

In [9]:
y_float = torch.randint(0, 7, (256,)).float()   # same labels, but now float32 - WRONG
logits = model(X)
loss = criterion(logits, y_float)               # this should error

RuntimeError: expected scalar type Long but found Float

### Ex 4 - learning rate matters
Rerun training (fresh model each time) with lr = 1e-3, then 1.0, then 1e-6.
Predict: what does a too-BIG lr do? A too-SMALL lr?

In [10]:
model = TinyNet()                                    # fresh desk every run!
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)   # try 1e-3, then 1.0, then 1e-6

X = torch.randn(256, 1, 28, 28)
y = torch.randint(0, 7, (256,))

for epoch in range(30):
    logits = model(X)
    loss = criterion(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  loss {loss.item():.4f}")

epoch  0  loss 2.0003
epoch  5  loss 1.2820
epoch 10  loss 0.7901
epoch 15  loss 0.4416
epoch 20  loss 0.2231
epoch 25  loss 0.1075


### result
- lr=1e-3: trains well (loss drops smoothly)
- lr=1.0: too big - steps overshoot, loss stays high / erratic
- lr=1e-6: too small - steps tiny, loss barely moves in 30 epochs
Learning rate = step size. Too big overshoots, too small crawls. It's the most-tuned hyperparameter.